In [81]:
import os, json, re, html, unicodedata
from pathlib import Path
import pandas as pd
from urllib.parse import urlparse

import sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("__file__"), "../..")))
import config  # expects BASE_URL, FANDOM_DATA_DIR, BASE_DIR

fandom_name = urlparse(config.BASE_URL).netloc.split(".")[0]
RAW_DATA_DIR = Path(config.FANDOM_DATA_DIR)
DATASET_DIR  = Path(config.BASE_DIR).parents[1] / "9.Span_Identification" / "datasets" / "processed"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
INPUT = RAW_DATA_DIR / f"master_spans_{fandom_name}.csv"
print("INPUT:", INPUT)
print("OUT  :", DATASET_DIR)

# --- Load (required columns) ---
REQ = ["article_id","paragraph_id","paragraph_text","start","end","link_text"]
df = pd.read_csv(INPUT, usecols=REQ)
print("Rows:", len(df))
print("Columns:", list(df.columns))
missing = set(REQ) - set(df.columns)
assert not missing, f"Missing: {missing}"

INPUT: /home/sundeep/Fandom-Span-Identification-and-Retrieval/1.Fandom_Dataset_Collection/raw_data/alldimensions_fandom_data/master_spans_alldimensions.csv
OUT  : /home/sundeep/Fandom-Span-Identification-and-Retrieval/9.Span_Identification/datasets/processed
Rows: 315635
Columns: ['article_id', 'paragraph_id', 'paragraph_text', 'link_text', 'start', 'end']


In [82]:
print(df.dtypes)
print(df[["paragraph_text","start","end","link_text"]].isna().sum())


article_id         int64
paragraph_id       int64
paragraph_text    object
link_text         object
start              int64
end                int64
dtype: object
paragraph_text     10
start               0
end                 0
link_text         744
dtype: int64


In [83]:
# coerce numerics, drop NaN offsets, fill text/link_text
df["start"] = pd.to_numeric(df["start"], errors="coerce")
df["end"]   = pd.to_numeric(df["end"],   errors="coerce")
df = df.dropna(subset=["start","end"]).copy()
df["start"] = df["start"].astype(int)
df["end"]   = df["end"].astype(int)
df["paragraph_text"] = df["paragraph_text"].fillna("").astype(str)
df["link_text"]      = df["link_text"].fillna("").astype(str)


In [84]:
# basic bounds/order checks
neg = (df["start"] < 0).sum()
rev = (df["end"] <= df["start"]).sum()
assert neg == 0, "Negative starts found"
assert rev == 0, "Non-positive span lengths found"
print("✓ load/clean OK")

✓ load/clean OK


In [85]:
# --- Helpers ---
def norm(s: str) -> str:
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKC", s)
    s = html.unescape(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def safe_slice(t: str, s: int, e: int) -> str:
    n = len(t); s = max(0, min(s, n)); e = max(0, min(e, n))
    return t[s:e] if e > s else ""

def try_fix_span(t: str, s: int, e: int, lt: str) -> tuple[int,int]:
    # exact raw first
    if safe_slice(t, s, e).strip() == lt.strip():
        return s, e
    n = len(t)
    # ±2 search on raw
    for ds in (-2,-1,0,1,2):
        for de in (-2,-1,0,1,2):
            ss, ee = max(0, s+ds), min(n, e+de)
            if ss < ee and safe_slice(t, ss, ee).strip() == lt.strip():
                return ss, ee
    # normalized fallback
    lt_n = norm(lt)
    for ds in (-2,-1,0,1,2):
        for de in (-2,-1,0,1,2):
            ss, ee = max(0, s+ds), min(n, e+de)
            if ss < ee and norm(safe_slice(t, ss, ee)) == lt_n:
                return ss, ee
    return s, e


In [86]:
# --- Sanity before fix ---
raw_slice = [safe_slice(t,s,e) for t,s,e in zip(df["paragraph_text"], df["start"], df["end"])]
mismatch0 = sum(a.strip()!=b.strip() for a,b in zip(raw_slice, df["link_text"]))
print("mismatches BEFORE fix:", mismatch0)

mismatches BEFORE fix: 785


In [87]:
# --- Apply offset auto-fix (±2 + norm) ---
df[["start","end"]] = [
    try_fix_span(t, s, e, lt)
    for t, s, e, lt in zip(df["paragraph_text"], df["start"], df["end"], df["link_text"])
]

# --- Sanity after fix ---
fix_slice = [safe_slice(t,s,e) for t,s,e in zip(df["paragraph_text"], df["start"], df["end"])]
mismatch1 = sum(a.strip()!=b.strip() for a,b in zip(fix_slice, df["link_text"]))
print("mismatches AFTER fix:", mismatch1)
assert mismatch1 <= mismatch0, "Auto-fix increased mismatches"


mismatches AFTER fix: 331


In [88]:
df2 = df.rename(columns={"paragraph_text":"text"})[["article_id","paragraph_id","text","start","end"]]
grouped = (df2.groupby(["article_id","paragraph_id","text"], as_index=False)
             .agg({"start": list, "end": list}))
grouped["spans"] = [[ [int(a),int(b)] for a,b in zip(st,en) ] for st,en in zip(grouped["start"], grouped["end"])]
grouped = grouped.drop(columns=["start","end"])

In [89]:
def validate_clip_dedup(text, spans):
    n=len(text); seen=set(); out=[]
    for a,b in spans:
        if a>b: a,b=b,a
        a=max(0,min(a,n)); b=max(0,min(b,n))
        if b-a>0 and (a,b) not in seen:
            seen.add((a,b)); out.append([a,b])
    return out

grouped["spans"] = [validate_clip_dedup(t,s) for t,s in zip(grouped["text"], grouped["spans"])]
grouped = grouped[grouped["spans"].str.len()>0].reset_index(drop=True)

In [91]:
# Safer check (no hard assert) + optional quick diagnostics
STRICT = False   # set True to enforce equality

links_before = len(df)  # each row ~= one link/span before grouping
spans_after  = sum(len(x) for x in grouped["spans"])

if spans_after == links_before:
    print("✓ grouping consistent")
elif STRICT:
    raise AssertionError(f"Lost/duplicated spans during grouping: {links_before} → {spans_after}")
else:
    print(f"⚠️ grouping changed count: {links_before} → {spans_after}")

    # minimal, fast diagnostics (comment out if not needed)
    try:
        exp = (df.rename(columns={"paragraph_text":"text"})
                 .groupby(["article_id","paragraph_id","text"]).size().rename("expected"))
        got = (grouped.assign(count=grouped["spans"].str.len())
                      .set_index(["article_id","paragraph_id","text"])["count"])
        mismatch = (got - exp).fillna(0)
        n_bad = int((mismatch != 0).sum())
        print(f"  paragraphs with mismatched counts: {n_bad}")
    except Exception as e:
        print(f"  (diag skipped: {e})")

⚠️ grouping changed count: 315635 → 157773
  paragraphs with mismatched counts: 28483


In [92]:
# Build a fresh grouped view from the current df (no dedup yet)
df2 = df.rename(columns={"paragraph_text":"text"})[["article_id","paragraph_id","text","start","end"]]
grouped_raw = (df2.groupby(["article_id","paragraph_id","text"], as_index=False)
                 .agg({"start": list, "end": list}))

links_before = len(df2)  # one row ~= one link
spans_after_raw = sum(len(st) for st in grouped_raw["start"])
print("links_before:", links_before, "| spans_after_raw:", spans_after_raw)  # should match

links_before: 315635 | spans_after_raw: 315635


In [93]:
# Count exact duplicate (start,end) pairs per paragraph
dup_total = 0
bad_total = 0
for st, en, text in zip(grouped_raw["start"], grouped_raw["end"], grouped_raw["text"]):
    pairs = [(int(a), int(b)) for a,b in zip(st,en)]
    dup_total += len(pairs) - len(set(pairs))          # duplicates
    bad_total += sum(1 for a,b in pairs if b <= a or a<0 or b>len(text))  # zero/invalid
print("duplicate spans:", dup_total)
print("invalid/zero-length spans:", bad_total)

duplicate spans: 157857
invalid/zero-length spans: 10


In [ ]:
def validate_clip_dedup(text, st, en):
    n = len(text)
    pairs = []
    for a,b in zip(st,en):
        a,b = int(a), int(b)
        if a>b: a,b = b,a
        a = max(0, min(a, n)); b = max(0, min(b, n))
        if b-a>0: pairs.append((a,b))
    # dedup
    pairs = list(dict.fromkeys(pairs))
    return [[a,b] for a,b in pairs]

grouped = grouped_raw.copy()
grouped["spans"] = [validate_clip_dedup(t, st, en)
                    for t, st, en in zip(grouped["text"], grouped["start"], grouped["end"])]

spans_after = sum(len(x) for x in grouped["spans"])
print("spans_after (post dedup/clip):", spans_after)
print("loss due to dedup/clip:", spans_after_raw - spans_after)

In [ ]:
EXPECTED = spans_after_raw   # compare to raw grouped count
ACTUAL   = spans_after
if ACTUAL == EXPECTED:
    print("✓ grouping consistent (pre→post)")
else:
    print(f"⚠️ count reduced by {EXPECTED - ACTUAL} (mostly duplicates/invalid spans removed)")